## Phase 0 — Drive, bundle hash, CUDA

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import hashlib, json, zipfile, subprocess, sys
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/gfm_r4_edge_acdc')
BUNDLE = DRIVE / 'gfm_r4_edge_acdc_colab_v1.zip'
SIDECAR = Path(str(BUNDLE) + '.sha256.json')
assert BUNDLE.is_file(), f'upload the bundle to {BUNDLE}'

expected = json.loads(SIDECAR.read_text())['sha256']
h = hashlib.sha256()
with BUNDLE.open('rb') as fh:
    for block in iter(lambda: fh.read(1 << 20), b''):
        h.update(block)
assert h.hexdigest() == expected, (h.hexdigest(), expected)
print('bundle sha256 OK')

REPO = Path('/content/gfm_r4_edge_acdc/repo')
REPO.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    archive.extractall(REPO.parent.parent)

manifest = json.loads((REPO / 'BUNDLE_MANIFEST.json').read_text())
print('seal check:', manifest['seal_check'])
print('files:', len(manifest['files']))

In [ ]:
import torch
assert torch.cuda.is_available(), 'no CUDA device - Runtime > Change runtime type > GPU'
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0))
print('free/total GiB:', [round(x / 2**30, 1) for x in torch.cuda.mem_get_info()])
TORCH_BEFORE = torch.__version__

In [ ]:
# Never reinstall torch - Colab's CUDA build must survive.
!pip -q install -r {REPO}/studies/splicebert_acceptor_coupling/colab_r4_package/requirements-colab.txt
import importlib, torch
importlib.reload(torch)
assert torch.__version__ == TORCH_BEFORE, 'torch was replaced; restart runtime and skip this cell'
print('deps installed, torch intact:', torch.__version__)

## Phase 1 — library tests on the GPU runtime

In [ ]:
import os
os.chdir(REPO)
# PYTHONPATH, not sys.path: `!python` is a SUBPROCESS and inherits the environment,
# not this kernel's sys.path.
!PYTHONPATH={REPO}/src python -m pytest tests/test_acdc_graph.py tests/test_acdc_kernel.py \
    tests/test_acdc_algorithm.py tests/test_acdc_device.py \
    tests/test_reverse_coupling_memorials.py tests/test_motif_cleanliness.py -q


## Phase 2 — cross-device floor. This sets the τ grid.

The committed reference panel was measured on the local device. The largest absolute
disagreement here **is** `device_floor`, and `min(τ) = 3 × floor`.

If this gate fails, the honest response is to **stop and report an instrument limit** — not to
lower the multiple. A τ below the floor yields a circuit that differs by accelerator with
nothing in the artifact disclosing it.

In [ ]:
MODEL = REPO / 'data/models/splicebert_official_archive/extracted/models/SpliceBERT.1024nt'
COHORT_N = 974          # R3's cohort size; keep both arms equal
FLOOR_N  = 32           # must match the committed reference

!python scripts/splicebert_acceptor_coupling/run_r4_paired_edge_acdc.py \
    --n {COHORT_N} --floor-n {FLOOR_N} --smoke --model {MODEL}

## Phase 3 — calibration: determinism and mode equivalence

In [ ]:
# A node-capped run exercises the full decision path cheaply. Its artifact is marked
# is_partial_sweep and is NOT a circuit.
OUT = DRIVE / 'calibration'
OUT.mkdir(parents=True, exist_ok=True)
!python scripts/splicebert_acceptor_coupling/run_r4_paired_edge_acdc.py \
    --n {COHORT_N} --floor-n {FLOOR_N} --idiom noising --max-nodes 4 \
    --model {MODEL} --out {OUT}/r4_calibration.json

## Phase 4 — benchmark. Read this before launching.

In [ ]:
# Projection from the MEASURED rate. The previous version read
# torch.cuda.max_memory_allocated() in this process while the runner is a
# subprocess, so it always reported 0 and its timer measured nothing.
MS_PER_EXAMPLE = 10.0e-3      # A100-80GB, batch 64, TF32 off (Phase 3, n=124)
EDGES = 4636

def hours(n, n_tau, idioms, n_dir=2):
    passes = sum(EDGES * (2 if i == 'meanabl' else 1) for i in idioms)
    return passes * n_dir * n_tau * n * MS_PER_EXAMPLE / 3600

for n in (128, 256, 944):
    for n_tau in (1, 2, 5):
        print(f"n={n:4d} tau={n_tau}  noising {hours(n,n_tau,['noising']):6.1f} h"
              f"   both {hours(n,n_tau,['noising','meanabl']):6.1f} h")
print()
print('mean-ablation costs 2x: the coupling is a difference of two readouts.')
print('If over budget, cut the tau grid or n EQUALLY on both directions --')
print('an asymmetric cut makes the paired partition uninterpretable.')


## Phase 5 — the target sweep

Set `LAUNCH_TARGET = True` below only after reading Phase 4's projection. `--resume` is
safe on a first run and resumes a disconnected run at the last completed node.


In [ ]:
LAUNCH_TARGET = False        # <- you set this, after reading Phase 4

COHORT_N = 128               # cut equally on both directions
TAU_HI   = 7.234911775011454e-04    # derived grid max  (300x the measured floor)
TAU_LO   = 7.234911775011454e-06    # derived grid min  (3x  the measured floor)
TAU      = TAU_HI            # leg 1; rerun with TAU_LO for leg 2
LEG      = 'hi'

TARGET = DRIVE / 'target'
STATE  = DRIVE / f'state_tau_{LEG}'
TARGET.mkdir(parents=True, exist_ok=True)
STATE.mkdir(parents=True, exist_ok=True)

if not LAUNCH_TARGET:
    raise SystemExit('LAUNCH_TARGET is False - nothing run. Set it to True to launch.')

# --resume is safe to pass on a first run: with no state file it starts clean.
# Re-running this cell after a disconnect picks up at the last finished node.
!python scripts/splicebert_acceptor_coupling/run_r4_paired_edge_acdc.py \
    --n {COHORT_N} --floor-n {FLOOR_N} --idiom both --mode sequential \
    --tau {TAU} --state-dir {STATE} --resume \
    --model {MODEL} --out {TARGET}/r4_tau_{LEG}.json \
    --gtf /content/drive/MyDrive/genomes/gencode.v50.basic.annotation.gtf.gz \
    --fasta /content/drive/MyDrive/genomes/hg38.fa


## Phase 6 — validate and return

In [ ]:
result = json.loads((TARGET / 'r4_tau_hi.json').read_text())
assert result['preflight_PASS'], 'preflight failed - do not report'
assert not result['is_partial_sweep'], 'partial sweep is not a circuit'
for key, part in result['partitions'].items():
    print(f"{key}: shared={part['n_shared']} fwd_only={part['n_forward_specific']} "
          f"rev_only={part['n_reverse_specific']} jaccard={part['jaccard']:.3f}")
print()
print('Copy back into results/splicebert_acceptor_coupling/ WITH its manifest.')
print('Report partitions PER IDIOM, never pooled (D9).')
